# Análisis de sesgos del corpus de recetas

Este cuaderno estudia posibles sesgos del corpus del proyecto RAG de recetas desde el contexto real del dominio: procedencia de las recetas, concentración temática, autores, ingredientes y posibles desbalances geográficos o de accesibilidad culinaria.

El objetivo no es demostrar que el corpus está "libre de sesgos" sino localizar concentraciones que puedan afectar al retrieval y a la generación, y proponer mitigaciones concretas para el proyecto.

In [1]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/xabierlosa/ProyectoPLN.git"
ROOT = Path("/content/ProyectoPLN")

if not (ROOT / ".git").exists():
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

sys.path.insert(0, str(ROOT))
print(f"Repositorio listo en {ROOT}")

Repositorio listo en /content/ProyectoPLN


## Qué sesgos se van a revisar

En este proyecto tiene sentido analizar principalmente: 
- Sesgo de origen o fuente: dominios o autores demasiado dominantes.
- Sesgo temático: recetas muy concentradas en unos pocos platos o ingredientes.
- Sesgo geográfico: presencia desigual de cocinas o lugares mencionados.
- Sesgo de accesibilidad: recetas que favorecen ingredientes caros o poco comunes.
- Sesgo de complejidad: exceso de recetas muy cortas o muy largas que distorsionen el retrieval.

Cuando el corpus no disponga de una columna explícita, el análisis se apoya en proxies razonables como `autor`, `titulo`, `ingredientes` e `instrucciones`.

In [ ]:
from pathlib import Path
import json
import math
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ChefGPT_Dataset_Random_Sample.json').exists():
            return candidate
    return start

RAW_PATH = ROOT / 'ChefGPT_Dataset_Random_Sample.json'
PREPROCESSED_PATH = ROOT / 'data' / 'preprocesado' / 'ChefGPT_Dataset_Preprocesado.json'
DATA_PATH = PREPROCESSED_PATH

print(f'ROOT: {ROOT}')
print(f'Dataset bruto: {RAW_PATH.exists()}')
print(f'Dataset preprocesado: {PREPROCESSED_PATH.exists()}')

ROOT: /content/ProyectoPLN
Dataset bruto: True
Dataset preprocesado: True


In [3]:
with DATA_PATH.open('r', encoding='utf-8') as handle:
    raw_recipes = json.load(handle)

df = pd.DataFrame(raw_recipes)
print(f'Registros: {len(df)}')
print(f'Columnas: {list(df.columns)}')
df.head(3)

Registros: 1500
Columnas: ['titulo', 'ingredientes', 'instrucciones', 'tiempo_total', 'porciones', 'autor', 'n_ingredientes']


,titulo,ingredientes,instrucciones,tiempo_total,porciones,autor,n_ingredientes
0,"Empanadillas de atún, tomate y huevo: un clási...","[300g Atún en conserva, 2 Huevo, Salsa de toma...",comenzamos cociendo los huevos en agua con sal...,15 minutos,4,Pakus,4
1,"Receta de caballa con salsa de tomate, la rece...","[2 Caballa, 400ml Salsa de tomate, 2 Patata, A...","en una cazuela baja, freímos ligeramente los l...",30 minutos,2,Pakus,6
2,Las naranjas están en su mejor momento: las me...,"[1 Pollo entero, 3 Naranja, 2 Patata, Finas Hi...","limpiamos el pollo de plumones, vigilad en esp...",90 minutos,4,Pakus,7


In [4]:
def first_existing(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None

author_col = first_existing(df.columns, ['autor', 'author', 'by', 'creator'])

print('author_col =', author_col)

def concentration_metrics(series: pd.Series) -> pd.DataFrame:
    counts = series.dropna().astype(str).str.strip()
    counts = counts[counts != '']
    if counts.empty:
        return pd.DataFrame([{'n_values': 0, 'top_share': np.nan, 'hhi': np.nan, 'entropy': np.nan}])
    freq = counts.value_counts()
    shares = freq / freq.sum()
    return pd.DataFrame([{
        'n_values': int(freq.shape[0]),
        'top_share': float(shares.iloc[0]),
        'hhi': float((shares ** 2).sum()),
        'entropy': float(-(shares * np.log2(shares)).sum()),
    }])

if author_col:
    author_stats = concentration_metrics(df[author_col])
    display(author_stats)
    display(df[author_col].fillna('Sin autor').astype(str).str.strip().replace('', 'Sin autor').value_counts().head(10))

author_col = autor


,n_values,top_share,hhi,entropy
0,26,0.180667,0.109731,3.591195


,count
autor,
Liliana Fuchs,271
Pakus,240
Carmen Tía Alia,226
Esther Clemente,146
Maria Jose,139
Mesa Cero Chefs y Jaime de las Heras,89
Inés Vazquez Noya,76
Unodedos,55
Marina Blanco,47


## TF-IDF sobre el corpus

El objetivo aquí es detectar vocabulario dominante y ver si el corpus se concentra demasiado en unos pocos ingredientes, técnicas o regiones culinarias. Eso ayuda a identificar sesgos temáticos y de accesibilidad.

In [5]:
SPANISH_STOPWORDS = {
    'a', 'acerca', 'ademas', 'ahi', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'asi', 'aun',
    'bajo', 'bien', 'cada', 'como', 'con', 'contra', 'cual', 'cuando', 'de', 'del', 'desde', 'donde',
    'e', 'el', 'ella', 'ellas', 'ellos', 'en', 'entre', 'era', 'erais', 'eran', 'eras', 'eres', 'es',
    'esa', 'esas', 'ese', 'esos', 'esta', 'estaba', 'estaban', 'estado', 'estais', 'estamos', 'estan',
    'estar', 'este', 'esto', 'estos', 'fue', 'fueron', 'ha', 'hace', 'hacer', 'han', 'hasta', 'hay',
    'la', 'las', 'lo', 'los', 'mas', 'me', 'mi', 'mis', 'mientras', 'muy', 'no', 'nos', 'nosotros',
    'o', 'os', 'otra', 'otros', 'para', 'pero', 'poco', 'por', 'porque', 'que', 'quien', 'quienes',
    'se', 'sea', 'si', 'sin', 'sobre', 'son', 'su', 'sus', 'tambien', 'te', 'tenia', 'tenian', 'tiene',
    'toda', 'todas', 'todo', 'todos', 'tu', 'tus', 'un', 'una', 'unas', 'uno', 'unos', 'ya',
}
TOKEN_RE = re.compile(r'[a-záéíóúñü]+', re.IGNORECASE)


def normalize_text(text):
    if text is None:
        return ''
    text = str(text).lower()
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def tokenize(text):
    tokens = TOKEN_RE.findall(normalize_text(text))
    return [token for token in tokens if len(token) > 2 and token not in SPANISH_STOPWORDS]


text_columns = [col for col in ['titulo', 'ingredientes', 'instrucciones'] if col in df.columns]
df['analysis_text'] = df[text_columns].fillna('').astype(str).agg(' '.join, axis=1)
df['tokens'] = df['analysis_text'].apply(tokenize)

doc_term_counts = [Counter(tokens) for tokens in df['tokens']]
doc_freq = Counter()
for tokens in df['tokens']:
    doc_freq.update(set(tokens))

n_docs = len(df)
idf = {term: math.log((1 + n_docs) / (1 + freq)) + 1 for term, freq in doc_freq.items()}


def top_tfidf_terms_for_indices(indices, top_n=12):
    aggregate = Counter()
    total_tokens = 0
    for idx in indices:
        counts = doc_term_counts[idx]
        aggregate.update(counts)
        total_tokens += sum(counts.values())
    if total_tokens == 0:
        return []
    scores = {}
    for term, count in aggregate.items():
        tf = count / total_tokens
        scores[term] = tf * idf.get(term, 0.0)
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_n]


overall_terms = top_tfidf_terms_for_indices(range(len(df)), top_n=20)
overall_terms_df = pd.DataFrame(overall_terms, columns=['termino', 'tfidf'])
display(overall_terms_df)


def group_tfidf_report(group_col, min_docs=10, top_groups=8):
    rows = []
    if group_col is None:
        return pd.DataFrame(rows)
    grouped = df.groupby(group_col).indices
    ranked_groups = sorted(grouped.items(), key=lambda item: len(item[1]), reverse=True)
    for group_value, indices in ranked_groups[:top_groups]:
        if len(indices) < min_docs:
            continue
        terms = top_tfidf_terms_for_indices(indices, top_n=8)
        rows.append({
            'grupo': str(group_value),
            'n_docs': len(indices),
            'top_terms': ', '.join([term for term, _ in terms]),
        })
    return pd.DataFrame(rows)

if author_col:
    display(group_tfidf_report(author_col))

,termino,tfidf
0,count,0.017113
1,aceite,0.017075
2,minutos,0.015801
3,azúcar,0.015732
4,sal,0.013902
5,salsa,0.013404
6,queso,0.013218
7,agua,0.012860
8,horno,0.012610
9,más,0.012462


,grupo,n_docs,top_terms
0,Liliana Fuchs,271,"añadir, count, dejar, más, aceite, sal, azúcar..."
1,Pakus,240,"salsa, pollo, carne, caldo, aceite, tomate, mi..."
2,Carmen Tía Alia,226,"count, queso, minutos, aceite, retiramos, dura..."
3,Esther Clemente,146,"azúcar, añadimos, crema, comenzaremos, horno, ..."
4,Maria Jose,139,"vel, azúcar, masa, count, thermomix, ponemos, ..."
5,Mesa Cero Chefs y Jaime de las Heras,89,"minutos, aceite, harina, masa, count, sal, agu..."
6,Inés Vazquez Noya,76,"aceite, pimienta, oliva, sal, cocinar, cebolla..."
7,Unodedos,55,"azúcar, count, masa, horno, echamos, leche, cr..."


**Interpretación de resultados:**

Los términos más relevantes del TF-IDF apuntan a un corpus muy centrado en cocina cotidiana y de uso general. Aparecen con fuerza palabras como `aceite`, `sal`, `agua`, `horno`, `harina`, `masa`, `pollo` o `cebolla`, lo que sugiere que el corpus está dominado por recetas domésticas, ingredientes comunes y técnicas básicas de preparación.

Esto no es un problema en sí, pero sí nos dice que el corpus puede estar sesgado hacia recetas populares y accesibles, con menor presencia de platos más especializados, regionales o de alta complejidad. En otras palabras, el sistema RAG tenderá a recuperar primero recetas muy generales y frecuentes, y podría dejar menos representadas cocinas minoritarias, ingredientes menos comunes o estilos culinarios más específicos.

Si en los grupos por autor o fuente aparecen términos distintos, eso indicaría además una concentración temática por procedencia: cada autor podría estar repitiendo su propio vocabulario o estilo culinario. En conjunto, el TF-IDF ayuda a ver que el sesgo principal aquí no parece demográfico, sino de cobertura del dominio y de repetición de patrones culinarios muy frecuentes.

## Topic modeling con LDA

Esta celda extrae temas latentes y ayuda a ver si el corpus está excesivamente concentrado en pocos patrones.

In [7]:
try:
    from sklearn.decomposition import LatentDirichletAllocation
    from sklearn.feature_extraction.text import CountVectorizer
except ImportError as exc:
    print(f'se omite LDA porque falta scikit-learn: {exc}')
else:
    vectorizer = CountVectorizer(
        max_df=0.85,
        min_df=2,
        ngram_range=(1, 2),
        stop_words=list(SPANISH_STOPWORDS),
    )
    X = vectorizer.fit_transform(df['analysis_text'])
    n_topics = min(6, max(2, X.shape[0] // 50))
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, learning_method='batch')
    topic_matrix = lda.fit_transform(X)
    feature_names = np.array(vectorizer.get_feature_names_out())

    def top_words_for_topic(topic_idx, top_n=10):
        component = lda.components_[topic_idx]
        return feature_names[np.argsort(component)[-top_n:][::-1]].tolist()

    topic_rows = []
    for topic_idx in range(n_topics):
        topic_rows.append({
            'topic_id': topic_idx,
            'top_words': ', '.join(top_words_for_topic(topic_idx, top_n=10)),
            'peso_medio': float(topic_matrix[:, topic_idx].mean()),
        })

    topics_df = pd.DataFrame(topic_rows)
    display(topics_df)

    dominant_topic = topic_matrix.argmax(axis=1)
    df['dominant_topic'] = dominant_topic
    topic_share = df['dominant_topic'].value_counts(normalize=True).sort_index()
    display(topic_share.to_frame('share'))

    if author_col:
        topic_by_author = pd.crosstab(df[author_col].fillna('Sin autor'), df['dominant_topic'])
        display(topic_by_author.head(10))

,topic_id,top_words,peso_medio
0,0,"crema, receta, azúcar, queso, chocolate, nata,...",0.108740
1,1,"azúcar, masa, harina, horno, mantequilla, minu...",0.212214
2,2,"aceite, oliva, sal, aceite oliva, pimienta, mi...",0.155997
3,3,"azúcar, leche, dejamos, vaso, receta, ponemos,...",0.076559
4,4,"aceite, minutos, salsa, sal, fuego, agua, rece...",0.255478
5,5,"minutos, sal, aceite, salsa, receta, 1count, p...",0.191012


,share
dominant_topic,
0,0.098667
1,0.213333
2,0.149333
3,0.068667
4,0.283333
5,0.186667


dominant_topic,0,1,2,3,4,5
autor,,,,,,
Anabel Palomares,0,0,0,0,1,0
Bea Orviz Tjiang,4,1,3,0,18,3
Carmen Tía Alia,34,40,9,19,44,80
Esther Clemente,29,53,10,6,28,20
Inés Vazquez Noya,7,8,24,1,15,21
Jaime de las Heras,2,0,2,1,8,3
Jesús Rojas,0,0,0,1,0,2
Juana Trujillo,0,0,0,0,3,1
Lena Álvarez,0,1,1,0,0,0


**Interpretación cualitativa del LDA:**

Los temas del LDA suelen confirmar que el corpus no está organizado en una gran variedad de dominios culinarios, sino en unos pocos bloques recurrentes de vocabulario. Cuando varios temas comparten palabras como `aceite`, `sal`, `horno`, `pollo`, `harina` o `cebolla`, eso indica que el corpus repite con mucha frecuencia el mismo tipo de recetas y técnicas, en vez de repartir el contenido entre cocinas o estilos muy distintos.

Desde el punto de vista de sesgos, esto sugiere un sesgo de cobertura y de concentración temática. El RAG tenderá a aprender y recuperar mejor las recetas más repetidas, rápidas o genéricas, mientras que las recetas menos frecuentes, más regionales o más complejas quedan diluidas en temas residuales. Si además uno de los temas domina claramente en proporción, el sesgo es todavía más claro: el corpus está sobreponderado hacia una familia concreta de preparaciones.

En resumen, el LDA no apunta tanto a un sesgo por autoría individual como a una falta de diversidad temática real. Eso es importante porque, aunque el sistema funcione bien sobre recetas comunes, puede responder peor cuando el usuario pida cocina regional, platos menos habituales o alternativas que se salgan del patrón dominante del corpus.

## Reconocimiento de entidades y huellas geográficas

El objetivo aquí no es etiquetar personas, sino detectar entidades mencionadas por el corpus que puedan introducir sesgo geográfico o de estilo editorial.

In [13]:
NER_AVAILABLE = False
ner_pipeline = None
try:
    from transformers import pipeline
except ImportError as exc:
    print(f'NER no disponible porque falta transformers: {exc}')
else:
    try:
        ner_pipeline = pipeline(
            'token-classification',
            model='Davlan/xlm-roberta-base-ner-hrl',
            aggregation_strategy='simple',
        )
        NER_AVAILABLE = True
        print('Modelo transformer multilingue cargado correctamente.')
    except Exception as exc:
        print('No se pudo cargar el modelo transformer multilingue.')
        print('Si quieres un modelo local ligero, puedes usar: python -m spacy download es_core_news_sm')
        print(f'Detalle: {exc}')

if NER_AVAILABLE and ner_pipeline is not None:
    if 'analysis_text' not in df.columns:
        text_columns = [col for col in ['titulo', 'ingredientes', 'instrucciones'] if col in df.columns]
        df['analysis_text'] = df[text_columns].fillna('').astype(str).agg(' '.join, axis=1)

    texts = df['analysis_text'].fillna('').astype(str)
    texts = texts[texts.str.strip() != '']
    total_texts = len(texts)

    if total_texts == 0:
        print('No hay textos para analizar en el corpus.')
    else:
        entity_counts = Counter()
        entity_label_counter = Counter()

        batch_size = 16
        max_chars = 512
        text_list = texts.tolist()

        for start in range(0, total_texts, batch_size):
            batch = [text[:max_chars] for text in text_list[start:start + batch_size]]
            try:
                batch_entities = ner_pipeline(batch)
            except Exception:
                continue

            if batch_entities and isinstance(batch_entities[0], dict):
                batch_entities = [batch_entities]

            for entities in batch_entities:
                for ent in entities:
                    label = ent.get('entity_group', ent.get('entity', 'UNK'))
                    entity_text = normalize_text(ent.get('word', ''))
                    if not entity_text:
                        continue
                    entity_counts[(label, entity_text)] += 1
                    entity_label_counter[label] += 1

        if entity_counts:
            rows = []
            for (label, entity_text), count in entity_counts.most_common(30):
                rows.append({'label': label, 'entidad': entity_text, 'frecuencia': count})
            ner_df = pd.DataFrame(rows)
            print(f'Entidades mas representativas (top {len(ner_df)}).')
            display(ner_df)
        else:
            print('No se detectaron entidades en el corpus.')
else:
    print('Se omite el analisis NER porque transformers o el modelo multilingue no estan disponibles en este entorno.')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: Davlan/xlm-roberta-base-ner-hrl
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo transformer multilingue cargado correctamente.
Entidades mas representativas (top 30).


,label,entidad,frecuencia
0,LOC,italia,4
1,LOC,oriente medio,4
2,LOC,mercado,3
3,ORG,na,3
4,PER,karlos arguiñano,3
5,LOC,cuaresma,3
6,LOC,semana santa,3
7,PER,francis paniego,2
8,LOC,españa,2
9,LOC,india,2


**Interpretación cualitativa (NER — corpus completo)**

- **Concentración de fuentes:** Predominan entidades de tipo `PER`, lo que sugiere una fuerte presencia de autores/fuentes recurrentes que pueden sesgar el lenguaje y el estilo editorial.
- **Poca huella geográfica:** La menor presencia de `LOC` indica referencias territoriales limitadas; el corpus puede carecer de representatividad regional.
- **Ruido y fragmentación:** En textos cortos algunas entidades aparecen fragmentadas o tokenizadas de forma irregular; interpretar frecuencias con cautela.
- **Impacto en RAG:** El retrieval puede priorizar contenido de las fuentes dominantes y reproducir su terminología, aumentando la homogenización de respuestas.

## Keyword Extraction por Keyness

Para este proyecto, el método más útil es **keyness** (log-odds con suavizado) porque permite comparar vocabulario entre grupos del corpus y detectar sesgos de cobertura de forma interpretable.

Aquí comparamos el autor más frecuente frente al resto del corpus. Los términos con keyness alta para ese autor indican posibles patrones editoriales o temáticos sobrerrepresentados.

In [12]:
import math
from collections import Counter

if not author_col:
    print('No hay columna de autor; no se puede calcular keyness por autor.')
else:
    # Texto base para extracción; reutiliza analysis_text si ya existe.
    if 'analysis_text' not in df.columns:
        text_cols = [c for c in ['titulo', 'ingredientes', 'instrucciones'] if c in df.columns]
        df['analysis_text'] = df[text_cols].fillna('').astype(str).agg(' '.join, axis=1)

    author_series = df[author_col].fillna('Sin autor').astype(str).str.strip().replace('', 'Sin autor')
    author_counts = author_series.value_counts()
    focus_author = author_counts.index[0]

    focus_idx = author_series[author_series == focus_author].index
    rest_idx = author_series[author_series != focus_author].index

    if len(focus_idx) < 10 or len(rest_idx) < 10:
        print('No hay suficientes documentos en los grupos para un cálculo estable de keyness.')
    else:
        focus_counter = Counter()
        rest_counter = Counter()

        for i in focus_idx:
            focus_counter.update(tokenize(df.at[i, 'analysis_text']))
        for i in rest_idx:
            rest_counter.update(tokenize(df.at[i, 'analysis_text']))

        vocab = set(focus_counter) | set(rest_counter)
        alpha = 0.5
        focus_total = sum(focus_counter.values())
        rest_total = sum(rest_counter.values())
        V = max(len(vocab), 1)

        rows = []
        for term in vocab:
            p_focus = (focus_counter.get(term, 0) + alpha) / (focus_total + alpha * V)
            p_rest = (rest_counter.get(term, 0) + alpha) / (rest_total + alpha * V)
            score = math.log(p_focus / p_rest)
            rows.append({
                'termino': term,
                'keyness': score,
                'freq_focus': int(focus_counter.get(term, 0)),
                'freq_rest': int(rest_counter.get(term, 0)),
            })

        keyness_df = pd.DataFrame(rows).sort_values('keyness', ascending=False).reset_index(drop=True)

        print(f'Autor foco (más frecuente): {focus_author} | docs foco={len(focus_idx)} | docs resto={len(rest_idx)}')
        print('Top términos sobrerrepresentados en el autor foco:')
        display(keyness_df.head(20))

        print('Top términos sobrerrepresentados en el resto del corpus:')
        display(keyness_df.sort_values('keyness', ascending=True).head(20))

Autor foco (más frecuente): Liliana Fuchs | docs foco=271 | docs resto=1229
Top términos sobrerrepresentados en el autor foco:


,termino,keyness,freq_focus,freq_rest
0,tamizar,5.247881,26,0
1,desechar,4.774096,16,0
2,reincorporar,4.573426,13,0
3,prefiere,4.322111,10,0
4,vieiras,4.322111,10,0
5,gurullos,4.222028,9,0
6,ñora,3.985639,7,0
7,igualando,3.985639,7,0
8,estragón,3.985639,7,0
9,desnatado,3.842538,6,0


Top términos sobrerrepresentados en el resto del corpus:


,termino,keyness,freq_focus,freq_rest
10858,escurrimos,-4.481313,0,158
10857,lavamos,-4.381893,0,143
10856,empezamos,-4.360766,0,140
10855,thermomix,-4.353623,0,139
10854,vel,-4.255801,0,126
10853,programamos,-4.215473,0,121
10852,añade,-4.015716,0,99
10851,ponemos,-3.942767,3,647
10850,batimos,-3.871842,1,258
10849,salteamos,-3.864075,0,85


**Interpretación cualitativa de Keyness:**

Los términos con keyness más alta para el autor foco indican qué vocabulario está sobrerrepresentado respecto al resto del corpus. Si aparecen ingredientes, técnicas o formatos muy repetidos en ese grupo, eso sugiere un sesgo de estilo editorial: parte del índice está capturando una forma concreta de escribir y cocinar, en lugar de reflejar toda la diversidad del dominio.

A la vez, los términos con keyness invertida (sobrerrepresentados en el resto) muestran qué temas quedan relativamente infraexpuestos en el autor dominante. Esta asimetría es útil para detectar sesgo de cobertura: el sistema puede recuperar con más facilidad recetas del patrón lexical dominante y penalizar recetas con vocabulario menos frecuente.

En términos de RAG, este resultado apunta a un riesgo claro de homogenización de respuestas. Aunque la recuperación funcione técnicamente, puede priorizar siempre las mismas familias de recetas, ingredientes o técnicas. La mitigación pasa por equilibrar fuentes/autores, diversificar muestreo y aplicar re-ranking con criterios de diversidad temática.

## Principales sesgos, efecto sobre RAG y mitigaciones

**Principales sesgos detectados**
- **Concentración por fuente/autor:** un reducido conjunto de autores domina una fracción relevante del corpus, lo que sesga la elección de vocabulario, formato y estilo editorial.
- **Cobertura temática limitada:** TF‑IDF y LDA muestran que el vocabulario y los temas tienden a centrarse en ingredientes y técnicas domésticas comunes (p. ej. aceite, sal, horno, pollo), con baja presencia de platos regionales o especializados.
- **Baja huella geográfica explícita:** pocas entidades `LOC` detectadas por NER, lo que indica escasa referencia territorial y posible infra‑representación de cocinas regionales.

En general, la baja huella geográfica y temática limitada no se consideraran sesgos perjudiciales, ya que se trata de un proyecto de PLN en Español y, consecuentemente, se espera una mayor representación de la comida tradicional española.

**Efecto esperado en un sistema RAG**
- **Recuperación sesgada:** los documentos de fuentes dominantes tenderán a aparecer en los primeros resultados, reduciendo la diversidad de respuestas.
- **Homogeneización de generación:** el generador probablemente reproduzca el estilo y el vocabulario de las fuentes dominantes, perdiendo variantes menos frecuentes.
- **Cobertura incompleta:** peticiones sobre cocina regional, ingredientes raros o formatos específicos pueden devolver resultados pobres o inexistentes.
- **Errores de señal por entidades fragmentadas:** entidades mal normalizadas pueden impedir re‑filtros útiles por lugar o ingrediente, degradando re‑ranking y agrupado semántico.


**Mitigaciones prácticas**
1. **Equilibrado del índice**: re‑muestreo (downsample) de autores dominantes antes de construir el índice para reducir la sobre‑representación.
2. **Metadata‑aware retrieval**: indexar y conservar metadatos (`autor`, `región`/`pais` cuando estén disponibles) y permitir filtros o boosting por diversidad/region en la etapa de retrieval.
3. **Re‑ranking por diversidad**: aplicar un paso de re‑ranking que combine relevancia con una métrica de diversidad (por autor, tema o geografía) para evitar monopolio de la top‑k inicial.
4. **Normalización y agrupación de entidades**: limpiar y consolidar entidades detectadas (listas de alias, lematización, reglas) para reducir fragmentación y habilitar filtros por `LOC` o `INGREDIENTE`.
6. **Pruebas y métricas**: introducir métricas de diversidad/recall por subgrupo (autor, región, tópico) y tests de regresión que vigilen el sesgo durante actualizaciones del índice.
7. **Prompting y post‑processing**: en generación, pedir explícitamente diversidad de fuentes en el prompt o penalizar repetición de frases estilo‑autor; usar templates que incentiven citar varias recetas o variantes.



**Siguiente paso:** calcular la distribución de autores y aplicar un downsample controlado sobre el 10–20% superior de autores dominantes; luego volver a indexar y medir impacto en diversidad del top‑10 de retrieval.

In [15]:
# Downsample controlado: reducir la representación del 10% superior de autores dominantes
from pathlib import Path
import math
import os

random_state = 42
out_dir = Path('data')
out_dir.mkdir(parents=True, exist_ok=True)

# Normalizar autor y contar
author_series = df[author_col].fillna('Sin autor').astype(str).str.strip().replace('', 'Sin autor')
author_counts = author_series.value_counts()

n_authors = len(author_counts)
top_k = max(1, math.ceil(0.10 * n_authors))
top_authors = author_counts.index[:top_k].tolist()

# Objetivo: limitar cada autor dominante a la mediana de documentos por autor (controlado)
cap_per_author = max(1, int(author_counts.median()))
print(f'Autores totales: {n_authors} | top_k={top_k} autores (top 10%)')
print(f'Cap por autor dominante: {cap_per_author} documentos')

# Seleccionar índices mantenidos
df['_author_norm'] = author_series
selected_indices = []
for author_value, group in df.groupby('_author_norm'):
    if author_value in top_authors:
        if len(group) > cap_per_author:
            sampled_idx = group.sample(n=cap_per_author, random_state=random_state).index.tolist()
            selected_indices.extend(sampled_idx)
        else:
            selected_indices.extend(group.index.tolist())
    else:
        selected_indices.extend(group.index.tolist())

new_df = df.loc[selected_indices].drop(columns=['_author_norm']).sort_index()

out_path = out_dir / 'ChefGPT_Dataset_downsampled_top10pct_authors.json'
new_df.to_json(out_path, orient='records', force_ascii=False)

print(f'Original: {len(df)} docs | Nuevo: {len(new_df)} docs | Archivo guardado: {out_path}')

Autores totales: 26 | top_k=3 autores (top 10%)
Cap por autor dominante: 20 documentos
Original: 1500 docs | Nuevo: 823 docs | Archivo guardado: data/ChefGPT_Dataset_downsampled_top10pct_authors.json
